In [ ]:
from langchain.agents import craeate_agent
from lanchain.tools import tool, ToolRuntime
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.types import Command
from pydantic import BaseModel
from dataclasses import dataclass
from langchain.agents.middleware import SummarizationMiddleware

system_prompt=""


@dataclass
class fav_col:
    fav_col: str = 'nothing'

@tools
def get_fav_col(runtime: ToolRuntime) -> str:
    """Get the favourite colour of the useer"""
    return runtime.context.favourite_colour

@tools 
def update_fav_col(favourite_colour:str, runtime:ToolRuntime) -> Command:
    """Update the favourite colour of the user in the state they have revealed it"""
    return Command(update={
        "favourite_colour": favourite_colour,
        "messages": [ToolMessage("Succesfully updated favourite colour", tool_call_id=runtime.tool_call_id)]})

@before_agent
def trim_message(state:AgentState, runtime:Runtime) -> dict[str, Any] | None:
    """Remove all the tool messages from the state"""
    messages = state["messages"]
    tool_messages = [m for m in messages if isinstance(m, HumanMessage)]
    return {"messages": [RemoveMessage(id=m.id) for m in tool_messages]}

class Ticket(BaseModel):

agent = craeate_agent(model='claude-sonnet-4-6',
                      system_prompt=system_prompt,
                      tools = ,
                      checkpointe=InMemorySaver(),
                      middleware=[
                          trim_messages
                      ]
                      context_schema=ColourContext
                      response_format=Ticket)

In [ ]:
from langchain_mcp_adapters.client import MultiServerMCPClient

client = MultiServerMCPClient(
    {
        "local_server": {
            "transport": "streamable_http",
            "command": "python",
        }
    }
)

In [ ]:
tools = await client.get_tools()
resources = await client.get_resources("local_server")
prompt = await client.get_prompt("local_server", "prompt")
prompt = prompt[0].content

In [ ]:
@dataclass
class LanguageContext:
    user_language: str = "English"

@dynamic_prompt
def user_language_prompt(request:ModelRequest) -> str:
    """Generate system prompt based on user role."""
    user_language = request.runtime.context.user_language
    base_prompt = "You are a helpful assistant"

    if user_language != "English":
        return f"{base_prompt} only respond in {user_language}"
    else:
        return base_prompt

In [ ]:
large_model = init_chat_model("claude-sonnet-4-6")
standard_model = init_chat_model("gpt-5-nano")

@wrap_model_call
def state_based_model(request:ModelRequest,
                      handler:[[ModelRequest], ModelResponse]) -> ModelResponse:
    """Select a model based on State conversation length"""
    
    message_count = len(request.messages)

    if message_count > 10:
        model = large_model
    else:
        model = standard_model

    request = request.override(tools=tools)
    return handler(request)

In [ ]:
@dataclass
class UserRole:
    user_role: str = "unauthorised"

@tool
def authenticate_user(query:str) -> str:
    """Searches the user in the database using SQL queries"""

    try:
        return db.run(query)
    except Exception as e:
        return f"Error: {e}"



@wrap_model_call
def dynamic_model_call(request:ModelRequest,
                       handler:Callable[[ModelRequest], ModelResponse]) -> ModelResponse:
    "Dynamically call tools based on the runtime context"

    user_role = request.runtime.context.user_role
    if user_role == "authorised":
        pass
    else:
        tools = [read_email, send_email]
        request = request.override(tools=tools)
    return handler(request)




In [ ]:
@dataclass
class UserRole:
    email_address: str = "example@example.com"
    password: str = "password"

from langchain.agents import AgentState

class AuthenticatedState(AgentState):
    authenticated: bool


In [ ]:
@tool
def check_inbox() -> str:
    """Check the inbox for recent emails"""
    return """Hi Julie,..."""

@tool 
def send_email() -> str:
    """Send a response email"""
    return "..."

@tool
def authenticate(email: str, password: str, runtime:ToolRuntime) -> Command:
    """Authenticate the user given the email and password"""
    if email==runtime.context.email_adress and password == runtime.context.password:
        return Command(update={
            "authenticated": True,
            "messages": [ToolMessage(
                "Succesfully authenticated",
                tool_call_id=runtime.tool_call_id
            )]
        })
    else:
        return Command(update={
            "authenticated": False,
            "messages": [ToolMessage(
                "Authentication failed",
                tool_call_id = runtime.tool_call_id
            )]
        })



In [ ]:
@wrap_model_call
def dynamic_tool_call(request: ModelRequest, handler: Callable([[ModelRequest], ModelResponse])) -> ModelResponse:
    """Allow read inbox and send emails only if users provide correct email and password"""
    authenticated = request.state.get("authenticated")
    if auhtenticated:
        tools = [check_inbox, read_email]
    else:
        tools = [authenticate]
    request = request.override(tools=tools)
    return handler(request)

authenticated_prompt = "You are a helpful assistant who can check inbox and send emails"
unauthenticated_prompt = "You are a helpful assistant that can authenticate users"

@dynamic_prompt
def dynamic_prompt(request:ModelRequest) -> str:
    """Generate system prompt based on authentication status"""
    authenticated = request.state.get("authenticated")
    if authenticated:
        return authenticated_prompt
    else:
        return unauthenticated_prompt
 

In [ ]:
agent = craeate_agent(
    model='claude-sonnet-4-6',
    tools = [authenticate, check_inbox, send_email],
    checkpointer = InMemorySaver,
    state_schema = AuthenticatedState,
    context_schema = EmailContext,
    middleware = [
        dynamic_tool_call,
        dynamic_prompt,
        HumanInTheLoopMiddleware(
            interrupt_on={
                "authenticate": False,
                "check_inbox": False,
                "send_email": True
}
        )
    ]
)